# P-Task1: Product Catalogue Cleaner

**Problem:** supplier ne raw product list bheji hai jis mein mixed types, duplicates aur missing values hain.
Mujhe data clean karna hai, price float mein convert karni hai, products ko category ke hisaab se
dictionary mein group karna hai aur end par neat summary print karni hai.

**Concepts used:** type casting, `isinstance()`, list / dict / set methods, string methods.

## 1. Raw data (jaisa supplier ne bheja)

In [1]:
raw_products = [
    {"id": 101, "name": "  Wireless Mouse ", "price": "1500", "category": "electronics"},
    {"id": 102, "name": "USB Cable", "price": 350, "category": "Electronics"},
    {"id": 103, "name": "Notebook A4", "price": "120.50", "category": "stationery"},
    {"id": 104, "name": "Ball Pen", "price": None, "category": "Stationery"},
    {"id": 105, "name": "Desk Lamp", "price": "Rs. 2,200", "category": "home"},
    {"id": 102, "name": "USB Cable", "price": 350, "category": "Electronics"},      # duplicate
    {"id": 106, "name": "", "price": "800", "category": "home"},                    # name missing
    {"id": "107", "name": "Water Bottle", "price": "650.0", "category": " home "},
    {"id": 108, "name": "Stapler", "price": "abc", "category": "stationery"},       # bad price
    {"id": 109, "name": "Keyboard", "price": 2999.99, "category": None},            # category missing
    {"id": 110, "name": "Highlighter", "price": True, "category": "stationery"},    # bool galat type
]
print("Total raw rows:", len(raw_products))

Total raw rows: 11


## 2. Helper functions
Pehle price ko float mein badalne ka function banaya. `Rs.` aur comma hata deta hai.
Agar convert na ho sake to `None` return karta hai.

In [2]:
def clean_price(value):
    # bool bhi int hota hai python mein, isliye pehle usay reject kiya
    if isinstance(value, bool) or value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        text = value.replace("Rs.", "").replace(",", "").strip()
        try:
            return float(text)
        except ValueError:
            return None
    return None


for sample in ["1500", 350, "120.50", "Rs. 2,200", "abc", None, True]:
    print(repr(sample), "->", clean_price(sample))

'1500' -> 1500.0
350 -> 350.0
'120.50' -> 120.5
'Rs. 2,200' -> 2200.0
'abc' -> None
None -> None
True -> None


## 3. Cleaning
Rules:
1. `id` ko int banao
2. name ke aage peeche se space hatao aur `.title()` lagao
3. category lower case mein karo
4. price float honi chahiye
5. duplicate id dobara nahi aani chahiye (set se check kiya)
6. name / price / category missing ho to row reject, reason ke saath

In [3]:
cleaned = []
rejected = []
seen_ids = set()

for row in raw_products:
    pid = int(row["id"])
    name = (row["name"] or "").strip().title()
    category = (row["category"] or "").strip().lower()
    price = clean_price(row["price"])

    if pid in seen_ids:
        rejected.append((pid, "duplicate"))
        continue
    if name == "":
        rejected.append((pid, "name missing"))
        continue
    if category == "":
        rejected.append((pid, "category missing"))
        continue
    if price is None:
        rejected.append((pid, "price invalid / missing"))
        continue

    seen_ids.add(pid)
    cleaned.append({"id": pid, "name": name, "price": price, "category": category})

print("Clean rows   :", len(cleaned))
print("Rejected rows:", len(rejected))
for pid, reason in rejected:
    print("  id", pid, "->", reason)

Clean rows   : 5
Rejected rows: 6
  id 104 -> price invalid / missing
  id 102 -> duplicate
  id 106 -> name missing
  id 108 -> price invalid / missing
  id 109 -> category missing
  id 110 -> price invalid / missing


## 4. Category ke hisaab se group karna

In [4]:
catalogue = {}
for item in cleaned:
    catalogue.setdefault(item["category"], []).append(item)

catalogue.keys()

dict_keys(['electronics', 'stationery', 'home'])

## 5. Summary

In [5]:
print("=" * 50)
print("PRODUCT CATALOGUE SUMMARY")
print("=" * 50)

for category, items in sorted(catalogue.items()):
    items.sort(key=lambda p: p["price"], reverse=True)
    total = sum(p["price"] for p in items)
    print(f"\n{category.upper()}  ({len(items)} products, total Rs {total:,.2f})")
    for p in items:
        print(f"   {p['id']:<5}{p['name']:<18}Rs {p['price']:>9,.2f}")

all_prices = [p["price"] for p in cleaned]
print("\n" + "-" * 50)
print("Total products :", len(cleaned))
print("Categories     :", len(catalogue))
print("Cheapest       :", min(cleaned, key=lambda p: p["price"])["name"])
print("Most expensive :", max(cleaned, key=lambda p: p["price"])["name"])
print(f"Average price  : Rs {sum(all_prices) / len(all_prices):,.2f}")

PRODUCT CATALOGUE SUMMARY

ELECTRONICS  (2 products, total Rs 1,850.00)
   101  Wireless Mouse    Rs  1,500.00
   102  Usb Cable         Rs    350.00

HOME  (2 products, total Rs 2,850.00)
   105  Desk Lamp         Rs  2,200.00
   107  Water Bottle      Rs    650.00

STATIONERY  (1 products, total Rs 120.50)
   103  Notebook A4       Rs    120.50

--------------------------------------------------
Total products : 5
Categories     : 3
Cheapest       : Notebook A4
Most expensive : Desk Lamp
Average price  : Rs 964.10


## Conclusion
Raw data mein 11 rows thin, jin mein se duplicate aur ghalat rows nikaal kar baqi data clean
dictionary mein aa gaya. Is task se maine type checking (`isinstance`), `try/except` se float conversion
aur `setdefault` se grouping seekhi.